# SAEBench: our SAE vs. the published baseline

This notebook analyses the SAEBench eval results produced by
`experiments/eval_saebench.sh` for **one SAE config** and compares them to the
**authors' published numbers** for the matching SAE Bench baseline — the config
you flagged on Neuronpedia,
[`gemma-2-2b/12-sae_bench_0125-batch_topk-res-64k__trainer_2`](https://www.neuronpedia.org/gemma-2-2b/12-sae_bench_0125-batch_topk-res-64k__trainer_2).

The authors' results come from the HuggingFace dataset
[`adamkarvonen/sae_bench_results_0125`](https://huggingface.co/datasets/adamkarvonen/sae_bench_results_0125),
which is the same data Neuronpedia surfaces.

**Trainer ↔ our member** (65k BatchTopK, mapped by the achieved L0 in the
authors' core results):

| trainer | author L0 | our member |
|---|---|---|
| trainer_0 | ~21  | `w65536_k20`  |
| trainer_1 | ~42  | `w65536_k40`  |
| **trainer_2** | **~84** | **`w65536_k80`** ← the config you sent |
| trainer_3 | ~169 | `w65536_k160` |
| trainer_4 | ~339 | `w65536_k320` |
| trainer_5 | ~696 | `w65536_k640` |

If you evaluated **multiple checkpoints** (`CHECKPOINTS=all`), the last section
plots each metric across training steps, with the author's final value drawn as
a reference line. (The authors only published a *final* checkpoint for
BatchTopK — their per-checkpoint series exist only for the TopK/Standard
variants — so the reference is a single horizontal line.)

## Runtime notes
- Run in the same ROCm container you train/eval in (`pip install -r requirements.txt`).
- This notebook is **read-only + a small HF download**: it does not run the base
  model, so it's cheap. Run `experiments/eval_saebench.sh` first to produce the
  results it reads.
- Downloading the authors' result JSONs needs internet + an HF token for gated
  access is **not** required (the results dataset is public).

## 1. Parameters

In [ ]:
from pathlib import Path

# --- Our results -------------------------------------------------------------
# The eval_saebench.sh output dir for the member you evaluated. By default that
# is <run>/saebench_eval/<member>; this notebook reads <OUR_OUTPUT_DIR>/eval_results/.
MEMBER = "w65536_k80"  # the 65k batch_topk trainer_2 config you sent
OUR_OUTPUT_DIR = Path(
    "/wekafs/smerrill/efficient_sae/experiments/results/saebench_gemma/saebench_eval"
) / MEMBER

# --- Authors' published baseline (HuggingFace) -------------------------------
AUTHOR_RESULTS_REPO = "adamkarvonen/sae_bench_results_0125"
# The 65k BatchTopK gemma-2-2b SAEs are published under the date-0108 release
# (the "0125" in the Neuronpedia id is the upload batch, not the results dir).
AUTHOR_RELEASE = "saebench_gemma-2-2b_width-2pow16_date-0108"
AUTHOR_ARCH = "BatchTopK"
AUTHOR_MODEL_TOKEN = "gemma-2-2b__0108"
AUTHOR_LAYER = 12

# k -> SAEBench trainer index, by achieved L0 (65k BatchTopK suite).
K_TO_TRAINER = {20: 0, 40: 1, 80: 2, 160: 3, 320: 4, 640: 5}

print(f"member       = {MEMBER}")
print(f"our results  = {OUR_OUTPUT_DIR / 'eval_results'}")
print(f"author repo  = {AUTHOR_RESULTS_REPO}")
print(f"author release = {AUTHOR_RELEASE}")

## 2. Imports & the metric registry

In [ ]:
import glob
import json
import re

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from huggingface_hub import hf_hub_download

pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda v: f"{v:.4f}")


def dig(d: dict, *path, default=None):
    '''Safely walk nested dicts: dig(metrics, "sparsity", "l0").'''
    cur = d
    for p in path:
        if not isinstance(cur, dict) or p not in cur:
            return default
        cur = cur[p]
    return cur


# Headline metrics per eval type: friendly_name -> (group, key, higher_is_better).
# These are extracted from eval_result_metrics in each result JSON.
METRIC_REGISTRY = {
    "core": {
        "L0":               ("sparsity", "l0", None),
        "CE_loss_score":    ("model_performance_preservation", "ce_loss_score", True),
        "KL_div_score":     ("model_behavior_preservation", "kl_div_score", True),
        "explained_var":    ("reconstruction_quality", "explained_variance", True),
        "frac_alive":       ("misc_metrics", "frac_alive", True),
    },
    "sparse_probing": {
        "probe_top1_acc":   ("sae", "sae_top_1_test_accuracy", True),
        "probe_top2_acc":   ("sae", "sae_top_2_test_accuracy", True),
        "probe_top5_acc":   ("sae", "sae_top_5_test_accuracy", True),
        "probe_full_acc":   ("sae", "sae_test_accuracy", True),
        "llm_top1_acc":     ("llm", "llm_top_1_test_accuracy", True),
    },
    "absorption": {
        "absorption_fraction": ("mean", "mean_absorption_fraction_score", False),
        "full_absorption":     ("mean", "mean_full_absorption_score", False),
        "num_split_features":  ("mean", "mean_num_split_features", False),
    },
    "scr": {
        "scr_thr_10": ("scr_metrics", "scr_metric_threshold_10", True),
        "scr_thr_20": ("scr_metrics", "scr_metric_threshold_20", True),
    },
    "tpp": {
        "tpp_thr_10": ("tpp_metrics", "tpp_threshold_10_total_metric", True),
        "tpp_thr_20": ("tpp_metrics", "tpp_threshold_20_total_metric", True),
    },
    "autointerp": {
        "autointerp_score": ("autointerp", "autointerp_score", True),
    },
    "ravel": {
        "disentanglement": ("ravel", "disentanglement_score", True),
        "cause":           ("ravel", "cause_score", True),
        "isolation":       ("ravel", "isolation_score", True),
    },
    "unlearning": {
        "unlearning_score": ("unlearning", "unlearning_score", True),
    },
}

HIGHER_IS_BETTER = {
    name: hib
    for et in METRIC_REGISTRY.values()
    for name, (_, _, hib) in et.items()
}


def extract_metrics(eval_type: str, result_json: dict) -> dict:
    '''Pull the headline metrics for one eval type out of a result JSON.'''
    m = result_json.get("eval_result_metrics", {})
    out = {}
    for name, (grp, key, _) in METRIC_REGISTRY.get(eval_type, {}).items():
        out[name] = dig(m, grp, key)
    return out

## 3. Load OUR eval results (all checkpoints)

In [ ]:
EVAL_DIR = OUR_OUTPUT_DIR / "eval_results"
assert EVAL_DIR.exists(), (
    f"{EVAL_DIR} not found. Run experiments/eval_saebench.sh for {MEMBER} first."
)

# eval_results/<eval_type>/<tag>_<member>_step_<STEP>_custom_sae_eval_results.json
step_re = re.compile(r"_step_(\d+)_")

rows = []
for path in sorted(glob.glob(str(EVAL_DIR / "*" / "*_eval_results.json"))):
    p = Path(path)
    eval_type = p.parent.name
    if eval_type not in METRIC_REGISTRY:
        continue
    sm = step_re.search(p.name)
    if sm is None or MEMBER not in p.name:
        continue  # not one of ours
    step = int(sm.group(1))
    data = json.loads(p.read_text())
    for metric, value in extract_metrics(eval_type, data).items():
        rows.append({"eval_type": eval_type, "metric": metric,
                     "step": step, "value": value})

ours_long = pd.DataFrame(rows)
assert not ours_long.empty, f"No parseable result JSONs under {EVAL_DIR}"

steps = sorted(ours_long["step"].unique())
final_step = max(steps)
print(f"Loaded {ours_long['eval_type'].nunique()} eval type(s) "
      f"across {len(steps)} checkpoint step(s): {steps}")
print("eval types:", sorted(ours_long['eval_type'].unique()))
ours_long.head(20)

## 4. Fetch the AUTHORS' published baseline

In [ ]:
trainer = K_TO_TRAINER[int(MEMBER.split("_k")[1])]
print(f"{MEMBER}  ->  {AUTHOR_RELEASE}  trainer_{trainer}")

author_rows = []
author_missing = []
for eval_type in sorted(ours_long["eval_type"].unique()):
    fname = (
        f"{eval_type}/{AUTHOR_RELEASE}/{AUTHOR_RELEASE}_{AUTHOR_ARCH}_"
        f"{AUTHOR_MODEL_TOKEN}_resid_post_layer_{AUTHOR_LAYER}_trainer_{trainer}"
        f"_eval_results.json"
    )
    try:
        local = hf_hub_download(AUTHOR_RESULTS_REPO, fname, repo_type="dataset")
        data = json.loads(Path(local).read_text())
    except Exception as e:  # noqa: BLE001
        author_missing.append((eval_type, str(e).splitlines()[0][:80]))
        continue
    for metric, value in extract_metrics(eval_type, data).items():
        author_rows.append({"eval_type": eval_type, "metric": metric, "value": value})

authors = pd.DataFrame(author_rows)
if author_missing:
    print("Could not fetch author results for:", author_missing)
print(f"\nFetched authors' trainer_{trainer} metrics for: "
      f"{sorted(authors['eval_type'].unique()) if not authors.empty else 'none'}")
authors

## 5. Head-to-head comparison (our final checkpoint vs. authors)

`delta = ours - authors`. The arrow shows whether our value moved in the
*better* direction for that metric (↑/↓ = better, — = worse/equal). L0 has no
"better" direction — interpret it as the achieved sparsity (should land near the
author's, ~84 for trainer_2).

In [ ]:
ours_final = (
    ours_long[ours_long["step"] == final_step]
    .set_index(["eval_type", "metric"])["value"]
)
auth_idx = authors.set_index(["eval_type", "metric"])["value"] if not authors.empty else pd.Series(dtype=float)

cmp_rows = []
for (eval_type, metric), ours_val in ours_final.items():
    auth_val = auth_idx.get((eval_type, metric))
    delta = (ours_val - auth_val) if (ours_val is not None and auth_val is not None) else None
    hib = HIGHER_IS_BETTER.get(metric)
    verdict = "—"
    if delta is not None and hib is not None and abs(delta) > 1e-9:
        better = (delta > 0) if hib else (delta < 0)
        verdict = "↑ better" if better else "↓ worse"
    cmp_rows.append({
        "eval_type": eval_type, "metric": metric,
        "ours": ours_val, "authors": auth_val, "delta": delta, "ours_vs_authors": verdict,
    })

comparison = pd.DataFrame(cmp_rows).set_index(["eval_type", "metric"])
print(f"Our checkpoint: step {final_step:,}   vs   authors' trainer_{trainer} (final)\n")
comparison

In [ ]:
# Grouped bar: ours vs authors for every shared metric (excludes L0, different scale).
plot_df = comparison.reset_index()
plot_df = plot_df[(plot_df["metric"] != "L0") & plot_df["authors"].notna() & plot_df["ours"].notna()]
if not plot_df.empty:
    melted = plot_df.melt(
        id_vars=["eval_type", "metric"], value_vars=["ours", "authors"],
        var_name="source", value_name="value",
    )
    melted["label"] = melted["eval_type"] + " · " + melted["metric"]
    fig = px.bar(melted, x="label", y="value", color="source", barmode="group",
                 title=f"{MEMBER} (ours, step {final_step:,}) vs authors' trainer_{trainer}",
                 text_auto=".3f")
    fig.update_layout(height=460, width=1000, xaxis_title="", yaxis_title="",
                      xaxis_tickangle=-35, legend_title="")
    fig.show()
else:
    print("No overlapping metrics to plot (did the author fetch succeed?).")

## 6. Metrics across training checkpoints

Only meaningful if you ran `CHECKPOINTS=all` (or a step list). Each metric is
plotted against the training step; the dashed line is the author's final value
for that metric (trainer_2), so you can see at which point in training our SAE
catches up to / matches the published baseline.

In [ ]:
if len(steps) <= 1:
    print(f"Only one checkpoint (step {final_step:,}) evaluated — re-run "
          f"eval_saebench.sh with CHECKPOINTS=all to populate this section.\n"
          f"(Requires a training run with intermediate checkpoints, e.g.\n"
          f"   N_CHECKPOINTS=10 WIDTHS=65536 KS=80 ./train_saebench.sh )")
else:
    metrics = list(ours_long[["eval_type", "metric"]].drop_duplicates().itertuples(index=False))
    for eval_type, metric in metrics:
        sub = (ours_long[(ours_long.eval_type == eval_type) & (ours_long.metric == metric)]
               .dropna(subset=["value"]).sort_values("step"))
        if sub.empty:
            continue
        fig = go.Figure()
        fig.add_trace(go.Scatter(x=sub["step"], y=sub["value"], mode="lines+markers",
                                 name="ours"))
        auth_val = auth_idx.get((eval_type, metric)) if not authors.empty else None
        if auth_val is not None:
            fig.add_hline(y=auth_val, line_dash="dash", line_color="firebrick",
                          annotation_text=f"authors trainer_{trainer} = {auth_val:.4f}",
                          annotation_position="top left")
        hib = HIGHER_IS_BETTER.get(metric)
        better = {True: " (higher better)", False: " (lower better)"}.get(hib, "")
        fig.update_layout(title=f"{eval_type} · {metric}{better}",
                          height=340, width=760,
                          xaxis_title="training step", yaxis_title=metric)
        fig.show()

## Notes
- **What "comparable" means:** `eval_saebench.py` uses SAEBench's own
  `run_all_evals_custom_saes.run_evals`, i.e. the *same* eval configs the authors
  used (core: 200 reconstruction / 2000 sparsity-variance batches on
  `Skylion007/openwebtext`, ctx 128, bf16). Remaining differences are training
  (we reimplement the recipe in SAELens, the authors used `dictionary_learning`),
  so small gaps are expected; large gaps are worth investigating.
- **Other configs:** point `MEMBER` at any `w<width>_k<k>` you evaluated and the
  trainer mapping + author release auto-adjust (re-derive `AUTHOR_RELEASE`'s
  `width-2pow..` if you change width).
- **More evals:** the comparison automatically covers whichever eval types you
  ran (`absorption`, `scr`, `tpp`, `autointerp`, `ravel`, `unlearning` are all in
  the registry). Run them via `EVALS=all ./eval_saebench.sh`.